# OPTIMIZACIÓN DE FLOORPLANNING (CVXPY) - PLANTA SIN TACC
## Modelo de 18 Sectores Nominales: Minimización de Costo de Manejo de Materiales (Z)

Este cuaderno implementa la metodología de **Floorplanning Convexo** basada en programación no lineal (CVXPY):
1. **18 Sectores Oficiales**: Dimensiones físicas exactas calculadas mediante el Método Guerchet, respetando las rotaciones a 90° del proyecto.
2. **Cero Pasillo Central Artificial**: Contacto rasante directo entre bloques departamentales ($\rho = 0.0\text{ m}$), ya que las áreas incluyen pasillos propios de evolución y circulación.
3. **Acoplamiento Rígido de Cadena Continua**: Moldeadora [5] $\rightarrow$ Horno Túnel [6] $\rightarrow$ Cinta Enfriado [7] alineados horizontalmente y en eje central.
4. **Cero Solapamientos Certificado**: Separación topológica estricta ($0.000000\text{ m}^2$) tanto en coordenadas continuas como tras redondeo a centímetros.
5. **Función Objetivo**: Minimización del **Costo de Manipulación de Materiales ($Z$)** ponderado por un factor de compacidad de la nave industrial envolvente ($\min Z + \gamma (W + H)$).

In [ ]:
import os
import json
import cvxpy as cp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.lines import Line2D

print("[OK] Librerías importadas exitosamente.")

### 1. Parámetros de Entrada (18 Sectores Nominales y Matriz de Flujos)

In [ ]:
BASE_DEPTS_V1 = {
    1:  {"name": "Depósito MP",             "w": 8.30,  "h": 6.30,  "area": 52.29,  "color": "#002060", "rot": 0, "desc": "Recepción y Almacén Materia Prima"},
    2:  {"name": "Aduana MP",               "w": 3.80,  "h": 3.80,  "area": 14.50,  "color": "#BDD7EE", "rot": 1, "desc": "Control de Ingreso y Desinfección"},
    3:  {"name": "Sección Pesado",          "w": 2.00,  "h": 3.80,  "area": 7.60,   "color": "#FCE4D6", "rot": 0, "desc": "Pesaje y Fraccionamiento MP"},
    4:  {"name": "Amasado G",               "w": 2.40,  "h": 2.17,  "area": 5.22,   "color": "#FFF2CC", "rot": 0, "desc": "Amasado Línea Galletitas"},
    5:  {"name": "Moldeado G",              "w": 2.50,  "h": 2.32,  "area": 5.80,   "color": "#FFF2CC", "rot": 0, "desc": "Moldeado Galletitas"},
    6:  {"name": "Horno Túnel G",           "w": 8.60,  "h": 2.30,  "area": 19.78,  "color": "#FCE4D6", "rot": 0, "desc": "Horno Túnel Continuo (8.6x2.3m)"},
    7:  {"name": "Cinta Enfriado G",        "w": 7.00,  "h": 1.19,  "area": 8.34,   "color": "#E2EFDA", "rot": 0, "desc": "Cinta de Enfriado Galletitas (7.0x1.2m)"},
    8:  {"name": "Envasado 1° Galletitas",  "w": 4.30,  "h": 4.00,  "area": 17.20,  "color": "#D9E1F2", "rot": 1, "desc": "Envasado Primario Galletitas"},
    9:  {"name": "Batido Panif.",            "w": 2.20,  "h": 1.85,  "area": 4.08,   "color": "#FFF2CC", "rot": 0, "desc": "Batido Línea Panificados"},
    10: {"name": "Dosificado P.",            "w": 2.00,  "h": 1.50,  "area": 3.00,   "color": "#FFF2CC", "rot": 0, "desc": "Dosificado Panificados"},
    11: {"name": "Fermentado Panes",         "w": 1.50,  "h": 1.23,  "area": 1.85,   "color": "#FFF2CC", "rot": 0, "desc": "Cámara Fermentación"},
    12: {"name": "Hornos Rotat. Panif.",     "w": 5.20,  "h": 3.74,  "area": 19.46,  "color": "#FCE4D6", "rot": 0, "desc": "Hornos Rotativos Panificados"},
    13: {"name": "Enfriado Panificados",     "w": 2.65,  "h": 2.00,  "area": 5.30,   "color": "#E2EFDA", "rot": 0, "desc": "Enfriado de Panificados"},
    14: {"name": "Envasado 1° Panif.",       "w": 4.80,  "h": 4.84,  "area": 23.23,  "color": "#D9E1F2", "rot": 1, "desc": "Envasado Primario Panificados"},
    15: {"name": "Depósito PT",             "w": 12.00, "h": 15.60, "area": 187.20, "color": "#1F4E79", "rot": 0, "desc": "Almacén Producto Terminado y Despacho"},
    16: {"name": "Lavado / Scrap",           "w": 2.90,  "h": 2.00,  "area": 5.80,   "color": "#EDEDED", "rot": 0, "desc": "Lavado de Bandejas y Scrap"},
    17: {"name": "Calidad Crudo",            "w": 1.50,  "h": 2.10,  "area": 3.15,   "color": "#EAEAEA", "rot": 1, "desc": "Laboratorio Control Calidad Crudo"},
    18: {"name": "Calidad Cocido",           "w": 4.50,  "h": 2.80,  "area": 12.60,  "color": "#EAEAEA", "rot": 0, "desc": "Laboratorio Control Calidad Cocido"},
}

COORDS_REF_V1 = {
    6:  [0.0, 0.0],
    5:  [-5.55, 0.0],
    7:  [7.80, 0.0],
    4:  [-5.55, 2.245],
    18: [7.80, -1.995],
    8:  [7.88, 3.11],
    15: [7.88, 12.91],
    3:  [-5.55, 5.23],
    2:  [-8.45, 5.23],
    16: [0.43, 7.98],
    14: [-0.52, 12.91],
    1:  [-8.45, 10.28],
    9:  [-3.45, 5.23],
    10: [-3.10, 3.55],
    13: [0.50, 5.97],
    17: [-7.50, 2.245],
    12: [0.50, 3.10],
    11: [-3.10, 2.19]
}

FLOW_DATA = {
    (1, 2): 1099.8, (2, 3): 1099.8,
    (3, 4): 600.0,  (4, 5): 600.0,  (5, 6): 600.0, (6, 7): 600.0, (7, 8): 600.0, (8, 15): 630.0,
    (3, 9): 600.0,  (9, 10): 600.0, (10, 11): 259.2, (10, 12): 345.6, (11, 12): 259.2,
    (12, 13): 604.8, (13, 14): 420.0, (14, 15): 500.0,
    (5, 4): 6.0, (7, 4): 22.0, (5, 16): 6.0, (7, 16): 5.5, (13, 16): 50.0,
    (4, 17): 2.0, (9, 17): 2.0, (7, 18): 3.0, (8, 18): 3.0, (13, 18): 3.0,
}

CONTINUOUS_CHAIN_PAIRS = set([(5, 6), (6, 7)])

df_depts = pd.DataFrame.from_dict(BASE_DEPTS_V1, orient='index')
print(f"Total sectores cargados: {len(df_depts)}")
df_depts[['name', 'area', 'w', 'h', 'rot', 'desc']]

### 2. Formulación y Resolución del Modelo de Floorplanning en CVXPY

In [ ]:
keys = sorted(list(BASE_DEPTS_V1.keys()))
n = len(keys)
idx = {k: i for i, k in enumerate(keys)}

w_val = np.array([BASE_DEPTS_V1[k]["w"] for k in keys], dtype=float)
h_val = np.array([BASE_DEPTS_V1[k]["h"] for k in keys], dtype=float)

# Determinar relaciones topológicas relativas (153 pares)
rel_type = {}
chain_pairs = set([(5, 6), (6, 5), (6, 7), (7, 6)])

for i in range(n):
    ki = keys[i]
    ci = COORDS_REF_V1[ki]
    wi, hi = w_val[i], h_val[i]
    for j in range(i + 1, n):
        kj = keys[j]
        if (ki, kj) in chain_pairs:
            continue
        cj = COORDS_REF_V1[kj]
        wj, hj = w_val[j], h_val[j]
        dx = cj[0] - ci[0]
        dy = cj[1] - ci[1]
        dist_x = abs(dx) / ((wi + wj) / 2.0)
        dist_y = abs(dy) / ((hi + hj) / 2.0)
        if dist_x >= dist_y:
            rel_type[(ki, kj)] = 'left' if dx >= 0 else 'right'
        else:
            rel_type[(ki, kj)] = 'below' if dy >= 0 else 'above'

# Variables del modelo
W = cp.Variable(shape=1, name="W")
H = cp.Variable(shape=1, name="H")
x = cp.Variable(shape=n, name="x")
y = cp.Variable(shape=n, name="y")

constraints = []

# A) Cadena Continua Rígida de Galletitas: [5] -> [6] -> [7]
i5, i6, i7 = idx[5], idx[6], idx[7]
constraints.append(x[i5] + w_val[i5] == x[i6])
constraints.append(x[i6] + w_val[i6] == x[i7])
constraints.append(y[i5] + h_val[i5] / 2.0 == y[i6] + h_val[i6] / 2.0)
constraints.append(y[i6] + h_val[i6] / 2.0 == y[i7] + h_val[i7] / 2.0)

# B) Separación estricta de pares (con holgura de 1 mm para robustez de redondeo)
delta_min = 0.001
for (ki, kj), r in rel_type.items():
    ii = idx[ki]
    ij = idx[kj]
    if r == 'left':
        constraints.append(x[ii] + w_val[ii] + delta_min <= x[ij])
    elif r == 'right':
        constraints.append(x[ij] + w_val[ij] + delta_min <= x[ii])
    elif r == 'below':
        constraints.append(y[ii] + h_val[ii] + delta_min <= y[ij])
    elif r == 'above':
        constraints.append(y[ij] + h_val[ij] + delta_min <= y[ii])

# C) Confinamiento en nave industrial
for i in range(n):
    constraints.append(x[i] >= 0)
    constraints.append(y[i] >= 0)
    constraints.append(x[i] + w_val[i] <= W)
    constraints.append(y[i] + h_val[i] <= H)

# D) Función Objetivo: Minimización de Manejo de Materiales (Z) + Compacidad
terms_Z = []
for (dept_a, dept_b), flow_val in FLOW_DATA.items():
    if (dept_a, dept_b) in CONTINUOUS_CHAIN_PAIRS:
        continue
    ia, ib = idx[dept_a], idx[dept_b]
    cxi = x[ia] + w_val[ia] / 2.0
    cyi = y[ia] + h_val[ia] / 2.0
    cxj = x[ib] + w_val[ib] / 2.0
    cyj = y[ib] + h_val[ib] / 2.0
    terms_Z.append(flow_val * (cp.abs(cxi - cxj) + cp.abs(cyi - cyj)))

Z_transporte = cp.sum(terms_Z)
gamma_nave = 100.0  # Ponderación bi-criterio para compacidad edilicia
objective = cp.Minimize(Z_transporte + gamma_nave * (W + H))

# Resolver problema convexo con solver de punto interior CLARABEL
prob = cp.Problem(objective, constraints)
prob.solve(solver=cp.CLARABEL)

print(f"Estado de optimización: {prob.status}")
print(f"Ancho total de nave (W) : {W.value[0]:.2f} m")
print(f"Largo total de nave (H) : {H.value[0]:.2f} m")
print(f"Superficie de nave (W*H): {W.value[0] * H.value[0]:.2f} m²")
print(f"Costo Z Operativo       : {Z_transporte.value:,.1f} kg*m/mes")

### 3. Auditoría Geométrica de Cero Solapamientos y Tabla de Coordenadas

In [ ]:
x_opt = np.array(x.value).flatten()
y_opt = np.array(y.value).flatten()

deptos_res = []
for idx_i, k in enumerate(keys):
    info = BASE_DEPTS_V1[k]
    deptos_res.append({
        "id": k,
        "name": info["name"],
        "desc": info["desc"],
        "color": info["color"],
        "area": round(info["area"], 2),
        "w": round(w_val[idx_i], 2),
        "h": round(h_val[idx_i], 2),
        "x": round(x_opt[idx_i], 2),
        "y": round(y_opt[idx_i], 2),
        "rot": info["rot"]
    })

# Auditoría geométrica exhaustiva
total_overlap = 0.0
conflictos = []
for i in range(n):
    di = deptos_res[i]
    xi, yi, wi, hi = di["x"], di["y"], di["w"], di["h"]
    for j in range(i + 1, n):
        dj = deptos_res[j]
        xj, yj, wj, hj = dj["x"], dj["y"], dj["w"], dj["h"]
        if set([di["id"], dj["id"]]) in [{5, 6}, {6, 7}]:
            continue
        ox = min(xi + wi, xj + wj) - max(xi, xj)
        oy = min(yi + hi, yj + hj) - max(yi, yj)
        if ox > 0.001 and oy > 0.001:
            total_overlap += ox * oy
            conflictos.append((di["id"], dj["id"], di["name"], dj["name"], ox * oy))

# Cálculo detallado de Z operativo y Z total
z_operativo = 0.0
z_total = 0.0
for (dept_a, dept_b), flow_val in FLOW_DATA.items():
    ia, ib = idx[dept_a], idx[dept_b]
    cxi = x_opt[ia] + w_val[ia] / 2.0
    cyi = y_opt[ia] + h_val[ia] / 2.0
    cxj = x_opt[ib] + w_val[ib] / 2.0
    cyj = y_opt[ib] + h_val[ib] / 2.0
    dist = abs(cxi - cxj) + abs(cyi - cyj)
    z_total += flow_val * dist
    if (dept_a, dept_b) not in CONTINUOUS_CHAIN_PAIRS:
        z_operativo += flow_val * dist

print("=" * 84)
print(f">>> SOLAPAMIENTO TOTAL: {total_overlap:.6f} m² <<<")
assert total_overlap < 1e-4, f"Error: Se detectaron solapamientos: {conflictos}"
print("[OK] CERO SOLAPAMIENTOS CERTIFICADO EXITOSAMENTE (0.000000 m²)")
print(f"Costo Transporte Operativo (Z) : {z_operativo:,.1f} kg*m/mes (Sin cinta acoplada)")
print(f"Costo Transporte Total (Z)     : {z_total:,.1f} kg*m/mes (Con cinta acoplada)")
print("=" * 84)

# Verificación de acoplamiento rígido de galletitas 5-6-7
d5 = next(d for d in deptos_res if d["id"] == 5)
d6 = next(d for d in deptos_res if d["id"] == 6)
d7 = next(d for d in deptos_res if d["id"] == 7)
print(f"Separación Moldeado -> Horno: {abs((d5['x'] + d5['w']) - d6['x']):.4f} m (Acople estricto)")
print(f"Separación Horno -> Cinta:    {abs((d6['x'] + d6['w']) - d7['x']):.4f} m (Acople estricto)")

df_res = pd.DataFrame(deptos_res)[['id', 'name', 'area', 'w', 'h', 'x', 'y', 'rot']]
df_res

### 4. Visualización Gráfica de la Planta Industrial Óptima

In [ ]:
W_val = round(float(W.value[0]), 2)
H_val = round(float(H.value[0]), 2)
dept_map = {d["id"]: d for d in deptos_res}

fig, ax = plt.subplots(figsize=(15, 14), dpi=150)

# Envolvente de Nave
nave_rect = patches.Rectangle((0, 0), W_val, H_val, linewidth=2.5, edgecolor="#002060", facecolor="#F8FAFC", linestyle="--", zorder=1)
ax.add_patch(nave_rect)

# Flujos de materiales con transparencia
mf = max(FLOW_DATA.values())
for (i, j), kg in FLOW_DATA.items():
    if (i, j) in [(5, 6), (6, 7)]:
        continue
    da, db = dept_map[i], dept_map[j]
    c1 = (da["x"] + da["w"] / 2.0, da["y"] + da["h"] / 2.0)
    c2 = (db["x"] + db["w"] / 2.0, db["y"] + db["h"] / 2.0)
    lw = 0.8 + (kg / mf) * 2.5
    ax.annotate('', xy=c2, xytext=c1,
                arrowprops=dict(arrowstyle="-|>", color='#002060', lw=lw, alpha=0.25, shrinkA=6, shrinkB=6), zorder=2)

# Dibujar cada bloque departamental
nombres_compactos = {
    11: "Ferment. Panes", 17: "Calidad Crudo", 10: "Dosificado P.",
    9:  "Batido Panif.",  13: "Enfriado Pan.", 16: "Lavado / Scrap",
    8:  "Envasado 1° Gall.", 14: "Envasado 1° Pan."
}

for d in deptos_res:
    xi, yi, wi, hi = d["x"], d["y"], d["w"], d["h"]
    color = d["color"]
    id_dep = d["id"]
    rot = d["rot"]
    
    edge_color = '#C00000' if id_dep in [5, 6, 7] else '#002060'
    edge_width = 2.2 if id_dep in [5, 6, 7] else 1.5
    
    rect = patches.Rectangle((xi, yi), wi, hi, facecolor=color, edgecolor=edge_color, linewidth=edge_width, zorder=3, alpha=0.90)
    ax.add_patch(rect)
    
    cx = xi + wi / 2.0
    cy = yi + hi / 2.0
    ax.plot(cx, cy, '+', ms=4.5, color='#C00000', zorder=5)
    
    is_dark = color in ['#002060', '#1F4E79']
    tc_title = 'white' if is_dark else '#002060'
    tc_sub = '#DDDDDD' if is_dark else '#444444'
    
    min_dim = min(wi, hi)
    fs = 5.2 if min_dim < 1.6 else (6.0 if min_dim < 2.2 else (8.5 if max(wi, hi) > 6.0 else 7.0))
    rot_str = " [R 90°]" if rot == 1 else ""
    label_nombre = nombres_compactos.get(id_dep, d["name"])
    
    if hi >= 2.0 and wi >= 2.2:
        ax.text(cx, cy + hi * 0.12, f"[{id_dep}] {label_nombre}{rot_str}", ha='center', va='center', fontsize=fs, fontweight='bold', color=tc_title, zorder=6)
        ax.text(cx, cy - hi * 0.15, f"{wi:.1f}×{hi:.1f}m ({d['area']:.1f}m²)", ha='center', va='center', fontsize=max(4.5, fs - 1.2), color=tc_sub, zorder=6)
    else:
        ax.text(cx, cy + hi * 0.10, f"[{id_dep}] {label_nombre}{rot_str}", ha='center', va='center', fontsize=fs, fontweight='bold', color=tc_title, zorder=6)
        ax.text(cx, cy - hi * 0.15, f"{wi:.1f}×{hi:.1f}m", ha='center', va='center', fontsize=max(4.5, fs - 1.0), color=tc_sub, zorder=6)

# Indicación limpia de acople continuo rígido en cadena de Galletitas [5-6-7]
d5, d6, d7 = dept_map[5], dept_map[6], dept_map[7]
cy_chain = d6["y"] + d6["h"] / 2.0
ax.annotate("", xy=(d6["x"] + 0.6, cy_chain), xytext=(d5["x"] + d5["w"] - 0.6, cy_chain),
            arrowprops=dict(arrowstyle="-|>", color="#C00000", lw=3.0, mutation_scale=16), zorder=8)
ax.annotate("", xy=(d7["x"] + 0.6, cy_chain), xytext=(d6["x"] + d6["w"] - 0.6, cy_chain),
            arrowprops=dict(arrowstyle="-|>", color="#C00000", lw=3.0, mutation_scale=16), zorder=8)

margin = 1.8
ax.set_xlim(-margin, W_val + margin)
ax.set_ylim(-margin, H_val + margin)
ax.set_aspect("equal")
ax.set_xlabel("Eje Longitudinal X [metros]", fontsize=11, fontweight="bold", color="#002060")
ax.set_ylabel("Eje Transversal Y [metros]", fontsize=11, fontweight="bold", color="#002060")
ax.set_title(f"DISTRIBUCIÓN EN PLANTA ÓPTIMA - FLOORPLANNING CONVEXO (CVXPY)\n18 Sectores Nominales | W = {W_val:.2f} m × H = {H_val:.2f} m | Z = {z_operativo:,.1f} kg·m/mes", fontsize=12, fontweight="bold", pad=15, color="#002060")

ax.annotate("", xy=(0, -0.9), xytext=(W_val, -0.9), arrowprops=dict(arrowstyle="<->", color="#C00000", lw=1.8))
ax.text(W_val / 2.0, -1.4, f"Ancho Total W = {W_val:.2f} m", ha="center", va="top", fontsize=10.5, fontweight="bold", color="#C00000")

ax.annotate("", xy=(-0.9, 0), xytext=(-0.9, H_val), arrowprops=dict(arrowstyle="<->", color="#C00000", lw=1.8))
ax.text(-1.4, H_val / 2.0, f"Largo Total H = {H_val:.2f} m", ha="right", va="center", rotation=90, fontsize=10.5, fontweight="bold", color="#C00000")

legend_elements = [
    patches.Patch(edgecolor="#002060", facecolor="#F8FAFC", linestyle="--", linewidth=2.0, label="Envolvente de Nave Industrial"),
    Line2D([0], [0], color='#C00000', lw=2.5, label='Cadena Galletitas [5-6-7] (Acople Físico Rasante)')
]
ax.legend(handles=legend_elements, loc="upper right", framealpha=0.95, fontsize=9.5)
ax.grid(True, linestyle=":", alpha=0.5, color="#BDC3C7")
plt.tight_layout()

os.makedirs("graficos", exist_ok=True)
plt.savefig("graficos/layout_floor_planning_18_sectores.png", dpi=300)
plt.show()

### 5. Exportación de Resultados (Excel y JSON)

In [ ]:
excel_path = "reporte_floor_planning_18_sectores.xlsx"
df_res.to_excel(excel_path, index=False)
print(f"[OK] Reporte Excel guardado en: {excel_path}")

json_path = "resultados_floor_planning_18_sectores.json"
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump({
        "num_sectores": 18,
        "W": W_val,
        "H": H_val,
        "area_nave": round(W_val * H_val, 2),
        "total_overlap": total_overlap,
        "z_operativo": round(z_operativo, 2),
        "z_total": round(z_total, 2),
        "departamentos": deptos_res
    }, f, indent=4, ensure_ascii=False)
print(f"[OK] Resultados JSON guardados en: {json_path}")